# 03 - Cell Type Annotation

**Goal:** turn anonymous 'cluster 0, 1, 2...' into real names like 'CD8 T cell'.

**Two approaches, used together:**
1. *Automated* with **celltypist** (a pre-trained PBMC model). This is the practical Python stand-in for Azimuth, which ships as an R/RDS object. If you (or a teammate) can run R, also map the Azimuth PBMC reference and compare — note any disagreements.
2. *Manual sanity-check* with canonical marker genes (defined in `config.py`). You confirm the automated labels light up the genes you'd expect.

Assigning a label per *cluster* (majority vote) is usually cleaner than per-cell.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # make the src/ package importable
from src import config as cfg
from src import utils as ut
ut.setup_scanpy()

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("scanpy", sc.__version__)

In [ ]:
import celltypist
from celltypist import models
adata = ut.load_checkpoint('02_clustered')

## Marker-gene sanity check first
Eyeball where canonical markers are expressed on the UMAP. This builds intuition before trusting any automated call.

In [ ]:
marker_flat = [g for genes in cfg.MARKER_GENES.values() for g in genes]
marker_flat = [g for g in marker_flat if g in adata.var_names]
sc.pl.umap(adata, color=marker_flat, ncols=4, save='_03_markers.png')

In [ ]:
sc.pl.dotplot(adata, cfg.MARKER_GENES, groupby='leiden',
              save='_03_marker_dotplot.png')

## Automated annotation with celltypist
Downloads a pre-trained immune model on first run. `majority_voting` smooths labels within each Leiden cluster.

In [ ]:
models.download_models(model=['Immune_All_Low.pkl'])
predictions = celltypist.annotate(adata, model='Immune_All_Low.pkl',
                                  majority_voting=True)
adata = predictions.to_adata()
adata.obs['cell_type'] = adata.obs['majority_voting']
sc.pl.umap(adata, color='cell_type', save='_03_celltypist.png')

## (Optional) collapse fine labels into the major lineages the rubric asks for
Edit this mapping after you see what celltypist returns, so downstream comparisons use clean categories (T / B / NK / Monocyte / DC ...).

In [ ]:
# Example — adjust keys to match the labels celltypist actually outputs:
lineage_map = {
    # 'CD4-positive, alpha-beta T cell': 'T cell',
    # 'CD8-positive, alpha-beta T cell': 'T cell',
    # 'Natural killer cell': 'NK cell',
    # 'Naive B cell': 'B cell',
    # 'Classical monocyte': 'Monocyte',
}
adata.obs['lineage'] = adata.obs['cell_type'].astype(str).replace(lineage_map)
adata.obs['lineage'].value_counts()

In [ ]:
ut.save_checkpoint(adata, '03_annotated')